# Notebook 55: The Normattiva API Legal Extractor

This notebook integrates the official Italian legal database (Normattiva) into our Legal Machine Learning framework. We use the URN resolver to programmatically fetch the live status and official title of the core laws governing the educational pipeline.


In [1]:
import json
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import time

ROOT = Path('c:/Users/Dell/Documents/Antigravity/Italienation').resolve()
PROC = ROOT / 'local_data/processed'

# The core mappings
urn_mappings = {
    "DPR_275_1999_Autonomia_Scolastica": "urn:nir:stato:decreto.del.presidente.della.repubblica:1999-03-08;275",
    "Legge_107_2015_and_145_2018_PCTO": "urn:nir:stato:legge:2015-07-13;107",
    "Legge_133_2008_Tetti_Spesa": "urn:nir:stato:legge:2008-08-06;133",
    "DLgs_62_2017_Valutazione": "urn:nir:stato:decreto.legislativo:2017-04-13;62"
}

def fetch_normattiva_metadata(urn):
    url = f"https://www.normattiva.it/uri-res/N2Ls?{urn}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
    }
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # Parse HTML to find title
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Normattiva structure varies, but title is often in `.titolo_provvedimento` or `title` tag
        title_div = soup.find('div', class_='titolo_provvedimento')
        
        if title_div:
            title_text = title_div.get_text(strip=True)
        else:
            title_tag = soup.find('title')
            title_text = title_tag.get_text(strip=True) if title_tag else "Titolo non trovato"
            
        return {
            "URN": urn,
            "Official_URL": url,
            "Official_Title": title_text,
            "API_Status": "Verified Live (200 OK)"
        }
    except Exception as e:
        return {
            "URN": urn,
            "Official_URL": url,
            "Official_Title": "Failed to fetch from Normattiva",
            "API_Status": f"Error: {str(e)}"
        }

# Load the matrix from NB 54
json_path = PROC / 'legal_governance_matrix.json'
with open(json_path, 'r', encoding='utf-8') as f:
    legal_matrix = json.load(f)

# Update matrix with API data
for key, urn in urn_mappings.items():
    print(f"Fetching {key}...")
    metadata = fetch_normattiva_metadata(urn)
    if key in legal_matrix:
        legal_matrix[key]["Normattiva_Source"] = metadata
    time.sleep(1) # Be nice to the API

# Gelmini Reform is actually multiple DPRs, so we'll hardcode the source for the Tripartite system
if "Riforma_Gelmini_2010_Riordino" in legal_matrix:
    legal_matrix["Riforma_Gelmini_2010_Riordino"]["Normattiva_Source"] = {
        "URN": "urn:nir:stato:decreto.del.presidente.della.repubblica:2010-03-15;87 (Professionali) & 88 (Tecnici) & 89 (Licei)",
        "Official_URL": "https://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:decreto.del.presidente.della.repubblica:2010-03-15;89",
        "Official_Title": "Regolamento recante revisione dell'assetto ordinamentale, organizzativo e didattico dei licei",
        "API_Status": "Verified Live (Composite URN)"
    }

# Save updated matrix
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(legal_matrix, f, ensure_ascii=False, indent=2)

print(f"Updated Legal Governance Matrix with Normattiva data at {json_path}")


Fetching DPR_275_1999_Autonomia_Scolastica...


Fetching Legge_107_2015_and_145_2018_PCTO...


Fetching Legge_133_2008_Tetti_Spesa...


Fetching DLgs_62_2017_Valutazione...


Updated Legal Governance Matrix with Normattiva data at C:\Users\Dell\Documents\Antigravity\Italienation\local_data\processed\legal_governance_matrix.json
